# Owned-buffer probe: izin diagnostik berbasis kontrol CPU yang ditinjau

Tes ownership native sebelumnya lulus. Persiapan berhenti karena 45 nilai fitur MusicFM120 melampaui toleransi FP32, walau seluruh label dan probabilitas akhir lulus. Notebook ini secara eksplisit memilih `owned-buffer-diagnostic` untuk melanjutkan eksperimen buffer yang sama. Default CLI tetap `strict-feature`.

Izin diagnostik hanya berlaku untuk quality report dan dua encoder yang persis sama dengan batch CPU enam run yang sudah ditinjau. Receipt kecil dalam repository mencatat hash batch sumber, sumber/model, serta ringkasan 129 keputusan yang diverifikasi ulang; tidak menyalin audio atau anotasi mentah. File receipt dipin SHA256. Jika model/quality/receipt berubah, persiapan ditolak.

`numerical_passed=false` dan `mac_feature_parity_passed=false` tetap dicatat ketika fitur berbeda. Label harus identik, data harus finite/valid, dan probabilitas base/konteks tetap wajib dalam batas absolut 0,0001. Mode ini mengizinkan probe terpisah, bukan sertifikasi ekuivalensi numerik atau integrasi SongMap. Readiness tetap false. Bobot 351,91 MiB, jadwal serial, window dan native ownership tests tetap.


In [ ]:
from pathlib import Path
from datetime import datetime
import hashlib,json,subprocess
ROOT=Path.cwd().resolve()
assert (ROOT/'scripts/prepare_canonical_pipeline_iphone.py').is_file(), 'Buka dari root SongMap'
SONGSTRUCT_ROOT=ROOT.parent/'Section-Annotations'
RUN_PYTHON=SONGSTRUCT_ROOT/'.venv-songformer311/bin/python'
assert RUN_PYTHON.is_file(), f'Python venv tidak ditemukan: {RUN_PYTHON}'
def sha(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()
EXPECTED_QUALITY='020f8d9a2bb5c29318f591a7c58d30b8f4cdc39eba47cf82c09820b538ac96fa'
QUALITY=next((p.resolve() for p in (ROOT/'outputs.local').glob('encoder-canonical-bank-*/quality/validation-all.json') if sha(p)==EXPECTED_QUALITY),None)
assert QUALITY, 'Report canonical full validation yang sudah ditinjau belum ditemukan'
q=json.loads(QUALITY.read_text())
assert q['selection_identity']=='d0aba90759d73e74dbc629950916531b90c6dfe219ac5d1f355667cc686944bf'
OUTPUT=ROOT/'outputs.local'/('canonical-owned-diagnostic-iphone-'+datetime.now().strftime('%Y%m%d-%H%M%S-%f'))
print('Quality:',QUALITY)
print('Proyek baru:',OUTPUT)

def run_logged(args,log):
    log=Path(log);log.parent.mkdir(parents=True,exist_ok=True)
    with log.open('w',encoding='utf-8') as handle:
        process=subprocess.Popen([str(v) for v in args],cwd=ROOT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in process.stdout:
                print(line,end='');handle.write(line);handle.flush()
            status=process.wait()
        except KeyboardInterrupt:
            process.terminate();process.wait();raise
    if status:raise RuntimeError(f'Persiapan gagal ({status}); lihat {log}. Simpan mac-preflight.json bila ada; gunakan OUTPUT baru untuk percobaan ulang.')


In [ ]:
run_logged([RUN_PYTHON,'-u',ROOT/'scripts/prepare_canonical_pipeline_iphone.py',
    '--songstruct-root',SONGSTRUCT_ROOT,'--songmap-root',ROOT,
    '--quality-report',QUALITY,'--preflight-policy','owned-buffer-diagnostic','--output-dir',OUTPUT],Path(str(OUTPUT)+'.prepare.log'))
proof=json.loads((OUTPUT/'preparation.json').read_text())
assert proof['native_typecheck_passed'] and proof['mac_pcm_pipeline_passed']
assert proof['quality_sha256']==EXPECTED_QUALITY and proof['canonical_encoder_policy']=='long-profile-shared-palette-v1'
assert proof['canonical_encoder_schedule']=='one-role-at-a-time-v1'
assert proof['tensor_transfer_policy']=='owned-fp32-between-models-v1' and proof['native_owned_tensor_self_test_passed']
assert proof['mac_preflight_policy']=='owned-buffer-diagnostic' and proof['mac_pcm_functional_passed']
print('Feature parity (diagnostic):',proof['mac_pcm_feature_parity_passed'])
print('CPU control receipt:',proof['cpu_control_receipt_sha256'])
print('Compiled encoder MiB:',proof['compiled_encoder_bytes']/2**20)
print('Seluruh model probe MiB:',proof['temporary_model_bytes']/2**20)
for check in proof['checks']:print(check['track_id'],'numerical:',check['numerical_passed'],'functional:',check['functional_passed'],check['native_context'])
subprocess.run(['open',str(OUTPUT/'Benchmark.xcodeproj')],check=True)


Stop run lama di Xcode. Pilih Team signing dan iPhone fisik, lalu jalankan aplikasi **Canonical Owned** dari proyek baru ini.

1. Jalankan **Raise a Hallelujah → CPU only → Run selected track**. Pastikan `native_owned_tensor_self_test_passed=true` dan `label_parity_passed=true`.
2. Jika lolos, Stop app dan Run kembali dari Xcode. Pilih **CPU + GPU candidate → Run 3 passes over all tracks**. Batch berhenti bila gate gagal. Tidak perlu mengulang batch CPU enam run yang lama.
3. Kirim JSON CPU kontrol dan `pipeline-batch-latest.json`. Jika gagal saat model load, simpan `pipeline-last-stage.json` dan log. `state=running` sendiri tidak membuktikan crash.

Aplikasi ini memakai container terpisah; laporan sebelumnya tetap ada di aplikasi Canonical Serial. Ini bukan izin integrasi atau bukti root cause. Tidak ada silent fallback CPU atau pelonggaran toleransi. Sampled footprint bukan peak memory, dan waktu probe dimulai dari PCM tanpa MP3 decoding/MusicUnderstanding/beat/chord/lyric.
